In [81]:
from neo4j import GraphDatabase
import pandas as pd

import json, pycountry


auth_data = pd.read_csv('../api_creds/neo4j_creds_gsv3.csv')

uri = auth_data['uri'].values[0]
username = auth_data['username'].values[0]
password = auth_data['password'].values[0]

driver = GraphDatabase.driver(uri, auth=(username, password))

# Setup

In [26]:
location = 'Oregon'
#location = 'all'

start_date = '2026-08-01'
end_date = '2026-08-31'

## GrooveSeeker Event Intelligence

Search structured event intelligence by location and date range. Use `all` to analyze the complete dataset across the past, present, or future.

```python
location = "all"
start_date = "2025-01-01"
end_date = "2027-12-31"
```

### Who should use it

* Venues, promoters, and event organizers
* Artists, managers, and booking agents
* Marketing and public relations agencies
* Media companies and local newsrooms
* Tourism, hospitality, and retail businesses
* Competitive intelligence and OSINT teams

### Why it matters

Instagram provides scattered posts shaped by accounts, followers, and algorithms. GrooveSeeker turns event information into structured datasets covering events, prices, entities, collaborations, websites, locations, and time.

Customers can study entire markets instead of individual posts. They can identify competitors, partners, publishers, pricing patterns, geographic opportunities, and collaboration networks while tracking how the event landscape changes across the past, present, and future.

This is event intelligence that can be searched, compared, analyzed, and turned into business decisions.


# tool 1: events

In [31]:
# genres = [
#     "alt-country",
#     "alternative country"
# ]

# pass in no genres to search for all genres
genres=[]

cypher_query = """

MATCH (e:Event)

WHERE (
    toLower($location) = 'all'
    OR EXISTS {
        MATCH (e)-[]-(l:Location)
        WHERE toLower(l.name) = toLower($location)
    }
)
AND e.event_dates[0] IS NOT NULL
AND date(e.event_dates[0]) >= date($start_date)
AND date(e.event_dates[0]) <= date($end_date)
AND (
    size($genres) = 0
    OR any(
        genre IN $genres
        WHERE toLower(e.event_text) CONTAINS toLower(genre)
    )
)

RETURN DISTINCT
    e.event_dates[0] AS event_date,
    e.event_title AS event_title,
    e.event_text AS event_text

ORDER BY event_date DESC
LIMIT 10000

"""

with driver.session() as session:
    data = json.loads(json.dumps(
        session.run(
            cypher_query,
            location=location,
            start_date=start_date,
            end_date=end_date,
            genres=genres or [],
        ).data(),
        default=str,
    ))

df = pd.DataFrame(data)
df.head(10)

,event_date,event_title,event_text
0,2026-08-31T20:00:00.000000000+00:00,BRENN! - AMATEUR AT BEST WORLD TOUR,BRENN! - AMATEUR AT BEST WORLD TOUR\n\nPerform...
1,2026-08-31T20:00:00.000000000+00:00,Brenn!,Brenn!\n\nPerformers:\nNot provided in source....
2,2026-08-31T20:00:00.000000000+00:00,"Brenn! - Hawthorne Theatre - Portland, OR","Brenn! - Hawthorne Theatre - Portland, OR\n\nP..."
3,2026-08-31T20:00:00.000000000+00:00,Thee Oh Sees in Portland,Thee Oh Sees in Portland\n\nPerformers:\nThee ...
4,2026-08-31T19:00:00.000000000+00:00,FREE! Covers Only,FREE! Covers Only\n\nPerformers:\nNot provided...
5,2026-08-31T19:00:00.000000000+00:00,Covers Only (Open Mic),Covers Only (Open Mic)\n\nPerformers:\nNot pro...
6,2026-08-31T19:00:00.000000000+00:00,Reading: Melissa Bird,Reading: Melissa Bird\n\nPerformers:\nDr. Meli...
7,2026-08-31T19:00:00.000000000+00:00,MODERN KUNDALINI (Monday Evenings),MODERN KUNDALINI (Monday Evenings) \n\nPerform...
8,2026-08-31T18:50:00.000000000+00:00,Solsara Practice Night,Solsara Practice Night\n\nPerformers:\nNot pro...
9,2026-08-31T18:30:00.000000000+00:00,*FREE* Live Music & Open Mic,*FREE* Live Music & Open Mic \n\nPerformers:\n...


In [32]:
df.shape

(1511, 3)

## Events

Search events by location and date range. Use `all` to search every location.

```python
location = "Oregon"
start_date = "2026-08-01"
end_date = "2026-08-31"
```

### Who should use it

* Local media and newsrooms
* Tourism and destination marketing organizations
* Marketing and event agencies
* Venues, promoters, and event organizers
* Hotels, restaurants, bars, and retailers
* Market researchers and OSINT analysts

### Why it matters

Instagram shows scattered posts from accounts you already know. This dataset provides a structured view of the broader event market.

Customers can track upcoming activity, find underserved opportunities, plan promotions, improve local coverage, and anticipate when events may create demand, traffic, or competition.


# tool 2: event_prices

In [35]:
# genres = [
#     "alt-country",
#     "alternative country"
# ]

# pass in no genres to search for all genres
genres=['punk']

cypher_query = """

MATCH (e:Event)
MATCH (e)-[:MENTIONS]-(entity:Entity)

WHERE (
    toLower($location) = 'all'
    OR EXISTS {
        MATCH (e)-[]-(l:Location)
        WHERE toLower(l.name) = toLower($location)
    }
)
AND e.event_dates[0] IS NOT NULL
AND date(e.event_dates[0]) >= date($start_date)
AND date(e.event_dates[0]) <= date($end_date)
AND e.event_text CONTAINS 'Price:'
AND NOT split(e.event_text, '\n')[9] CONTAINS 'Not provided in source.'
AND (
    size($genres) = 0
    OR any(
        genre IN $genres
        WHERE toLower(e.event_text) CONTAINS toLower(genre)
    )
)

WITH
    split(e.event_text, '\n')[0] AS event_title,
    e.url AS url,
    head(collect(DISTINCT e.event_dates[0])) AS event_date,
    head(collect(DISTINCT split(e.event_text, '\n')[9])) AS event_price,
    collect(DISTINCT entity.name) AS entities

RETURN
    event_date,
    event_title,
    event_price,
    entities

ORDER BY event_date DESC
LIMIT 10000

"""

with driver.session() as session:
    data = json.loads(json.dumps(
        session.run(
            cypher_query,
            location=location,
            start_date=start_date,
            end_date=end_date,
            genres=genres or [],
        ).data(),
        default=str,
    ))

df = pd.DataFrame(data)
df.head(10)

# I will use simple prompt-engineering to extract cleaned up event_prices
# This is raw data that leads to a premium dataset

,event_date,event_title,event_price,entities
0,2026-08-28T23:55:00.000000000+00:00,HOTWAX disco~glam~metal~punk~hiphop,$2.08,"[Dante's Live, HOTWAX]"
1,2026-08-16T21:00:00.000000000+00:00,"Now, That's What I Call Goth",$5 Cover,"[The Coffin Club, DJ F]"
2,2026-08-16T20:30:00.000000000+00:00,"Street Play(LA), Black Ends(Sea), Zailee Haze,...",USD 15,"[Black Ends, Filling, Shanghai Tunnel Bar & Ve..."
3,2026-08-15T18:30:00.000000000+00:00,Beach Bunny & The Beths: US Tour 2026,$64.46,"[The Beths, Beach Bunny]"
4,2026-08-15T12:00:00.000000000+00:00,Rockaway Beach Music Festival 2026,Price:,"[Kris Stuart, Mike Mannequin, The Upkeeps, Okl..."
5,2026-08-15T12:00:00.000000000+00:00,Summer Shred Music Festival,Free,"[metal, rock, Summer Shred Music Festival, punk]"
6,2026-08-12T22:00:00.000000000+00:00,Oubliette Old School Night No Cover - The Coffin,No Cover,"[DJ NoN, The Coffin Club, DJ John the Revelato..."
7,2026-08-09T18:30:00.000000000+00:00,LCD Soundsystem,$92.50 to $104.50,"[LCD Soundsystem, McMenamins Edgefield, Victor..."
8,2026-08-09T18:00:00.000000000+00:00,Radioactive Ball welcomes the Vandoliers to Ba...,"$25 in advance, $30 at the door","[The Vandoliers, Radioactive Ball, KBZR]"
9,2026-08-09T18:00:00.000000000+00:00,Radioactive Ball welcomes the Vandoliers to Ba...,"$25 in advance, $30 at the door","[Vandoliers, Churchill School, Radioactive Bal..."


In [36]:
df.shape

(12, 4)

## Event Prices

Search event prices by location and date range. Use `all` to compare prices across every location.

```python
location = "Oregon"
start_date = "2026-08-01"
end_date = "2026-08-31"
```

### Who should use it

* Venues and event promoters
* Artists, booking agents, and managers
* Ticketing and event platforms
* Marketing and event agencies
* Tourism and hospitality businesses
* Market researchers and OSINT analysts

### Why it matters

Instagram can show what individual events charge, but it cannot easily reveal pricing patterns across thousands of events.

Customers can benchmark competitors, compare markets, study free versus paid events, identify pricing gaps, and make better decisions about tickets, promotions, and positioning.


# tool 3: locations

In [7]:
cypher_query = """ 
 
MATCH (l:Location) 
 
WHERE l.name =~ '^[A-Za-z].*'
AND (
    toLower($location) = 'all'
    OR toLower(l.name) CONTAINS toLower($location)
)
 
RETURN DISTINCT l.name AS location 
 
ORDER BY location 
LIMIT 10000
 
""" 
 
with driver.session() as session: 
    data = json.loads(json.dumps(session.run(cypher_query, location=location).data(), default=str)) 
    
df = pd.DataFrame(data) 
df.head(10)

,location
0,Central Oregon
1,Eastern Oregon
2,Oregon
3,Oregon City
4,Oregon Coast
5,Oregon Park
6,Oregon Ridge
7,Oregon State Fair
8,Oregon State University
9,Oregon Zoo


In [8]:
df.shape

(12, 1)

## Locations

Search available locations by name. Use `all` to return every location in GrooveSeeker.

```python
location = "Oregon"
```

### Who should use it

* Marketing and event agencies
* Tourism organizations
* Local media and newsrooms
* Venue and promoter networks
* Businesses planning geographic expansion
* Market researchers and OSINT analysts

### Why it matters

Instagram organizes discovery around accounts and engagement, not complete geographic coverage.

Customers can identify available markets, discover locations they may have overlooked, compare geographic opportunities, and select locations for deeper event research.


# tool 4: Entities

In [41]:
# genres = [
#     "alt-country",
#     "alternative country"
# ]

# pass in no genres to search for all genres
genres=[]

cypher_query = """

MATCH (entity:Entity)-[:MENTIONS]-(e:Event)

WHERE (
    toLower($location) = 'all'
    OR EXISTS {
        MATCH (e)-[]-(l:Location)
        WHERE toLower(l.name) = toLower($location)
    }
)
AND e.event_dates[0] IS NOT NULL
AND date(e.event_dates[0]) >= date($start_date)
AND date(e.event_dates[0]) <= date($end_date)
AND (
    size($genres) = 0
    OR any(
        genre IN $genres
        WHERE toLower(e.event_text) CONTAINS toLower(genre)
    )
)

RETURN DISTINCT
    e.event_dates[0] AS event_date,
    entity.name AS entity

ORDER BY event_date DESC
LIMIT 10000

"""

with driver.session() as session:
    data = json.loads(json.dumps(
        session.run(
            cypher_query,
            location=location,
            start_date=start_date,
            end_date=end_date,
            genres=genres or [],
        ).data(),
        default=str,
    ))

df = pd.DataFrame(data)
df.head(10)

,event_date,entity
0,2026-08-31T20:00:00.000000000+00:00,Hawthorne Theatre
1,2026-08-31T20:00:00.000000000+00:00,BRENN!
2,2026-08-31T20:00:00.000000000+00:00,Brenn
3,2026-08-31T20:00:00.000000000+00:00,Brenn!
4,2026-08-31T20:00:00.000000000+00:00,Tom Siletto
5,2026-08-31T20:00:00.000000000+00:00,McMenamins Crystal Ballroom
6,2026-08-31T20:00:00.000000000+00:00,Thee Oh Sees
7,2026-08-31T19:00:00.000000000+00:00,Atlantis Lounge
8,2026-08-31T19:00:00.000000000+00:00,Mississippi Pizza
9,2026-08-31T19:00:00.000000000+00:00,Oregon


In [42]:
df.shape

(3659, 2)

## Entities

Search entities connected to events within a location and date range. Entities may include venues, artists, organizations, and other names associated with an event. Use `all` to search every location.

```python
location = "Oregon"
start_date = "2026-08-01"
end_date = "2026-08-31"
```

### Who should use it

* Marketing and event agencies
* Venues and promoters
* Artists, managers, and booking agents
* Local media and newsrooms
* Market researchers and OSINT analysts

### Why it matters

Instagram limits discovery to accounts, hashtags, and recommendations. This dataset reveals the broader group of entities participating in a market.

Customers can discover active venues and artists, identify potential partners or competitors, track who is appearing in a location, and find entities outside their existing network.


# tool 5: Entities Events

In [43]:
# genres = [
#     "alt-country",
#     "alternative country"
# ]

# pass in no genres to search for all genres
genres=[]

cypher_query = """

MATCH (entity:Entity)-[:MENTIONS]-(e:Event)

WHERE (
    toLower($location) = 'all'
    OR EXISTS {
        MATCH (e)-[]-(l:Location)
        WHERE toLower(l.name) = toLower($location)
    }
)
AND e.event_dates[0] IS NOT NULL
AND date(e.event_dates[0]) >= date($start_date)
AND date(e.event_dates[0]) <= date($end_date)
AND (
    size($genres) = 0
    OR any(
        genre IN $genres
        WHERE toLower(e.event_text) CONTAINS toLower(genre)
    )
)

RETURN DISTINCT
    e.event_dates[0] AS event_date,
    entity.name AS entity,
    split(e.event_text, '\n')[0] AS event_title

ORDER BY event_date DESC
LIMIT 10000

"""

with driver.session() as session:
    data = json.loads(json.dumps(
        session.run(
            cypher_query,
            location=location,
            start_date=start_date,
            end_date=end_date,
            genres=genres or [],
        ).data(),
        default=str,
    ))

df = pd.DataFrame(data)
df.head(10)

,event_date,entity,event_title
0,2026-08-31T20:00:00.000000000+00:00,Hawthorne Theatre,BRENN! - AMATEUR AT BEST WORLD TOUR
1,2026-08-31T20:00:00.000000000+00:00,BRENN!,BRENN! - AMATEUR AT BEST WORLD TOUR
2,2026-08-31T20:00:00.000000000+00:00,Hawthorne Theatre,Brenn!
3,2026-08-31T20:00:00.000000000+00:00,Brenn,Brenn!
4,2026-08-31T20:00:00.000000000+00:00,Hawthorne Theatre,"Brenn! - Hawthorne Theatre - Portland, OR"
5,2026-08-31T20:00:00.000000000+00:00,Brenn!,"Brenn! - Hawthorne Theatre - Portland, OR"
6,2026-08-31T20:00:00.000000000+00:00,Tom Siletto,"Brenn! - Hawthorne Theatre - Portland, OR"
7,2026-08-31T20:00:00.000000000+00:00,McMenamins Crystal Ballroom,Thee Oh Sees in Portland
8,2026-08-31T20:00:00.000000000+00:00,Thee Oh Sees,Thee Oh Sees in Portland
9,2026-08-31T19:00:00.000000000+00:00,Atlantis Lounge,FREE! Covers Only


In [44]:
df.shape

(3880, 3)

## Entity Events

Connect entities to the specific events in which they appear. Use `all` to search every location.

```python
location = "Oregon"
start_date = "2026-08-01"
end_date = "2026-08-31"
```

### Who should use it

* Venues and event promoters
* Artists, managers, and booking agents
* Marketing and event agencies
* Local media and newsrooms
* Market researchers and OSINT analysts

### Why it matters

Instagram may reveal individual announcements, but it does not provide a structured view of which entities are connected to which events.

Customers can track entity activity, study venue and artist schedules, identify recurring relationships, discover potential partners, and analyze how participants move through an event market.


# tool 6: Entities Events Prices

In [49]:
# genres = [
#     "alt-country",
#     "alternative country"
# ]

# pass in no genres to search for all genres
genres=[]

cypher_query = """

MATCH (entity:Entity)-[:MENTIONS]-(e:Event)

WHERE (
    toLower($location) = 'all'
    OR EXISTS {
        MATCH (e)-[]-(l:Location)
        WHERE toLower(l.name) = toLower($location)
    }
)
AND e.event_dates[0] IS NOT NULL
AND date(e.event_dates[0]) >= date($start_date)
AND date(e.event_dates[0]) <= date($end_date)
AND e.event_text CONTAINS 'Price:'
AND NOT split(e.event_text, '\n')[9] CONTAINS 'Not provided in source.'
AND (
    size($genres) = 0
    OR any(
        genre IN $genres
        WHERE toLower(e.event_text) CONTAINS toLower(genre)
    )
)

RETURN DISTINCT
    e.event_dates[0] AS event_date,
    entity.name AS entity,
    split(e.event_text, '\n')[0] AS event_title,
    split(e.event_text, '\n')[9] AS event_price

ORDER BY event_date DESC
LIMIT 10000

"""

with driver.session() as session:
    data = json.loads(json.dumps(
        session.run(
            cypher_query,
            location=location,
            start_date=start_date,
            end_date=end_date,
            genres=genres or [],
        ).data(),
        default=str,
    ))

df = pd.DataFrame(data)
df.head(10)

# I will use simple prompt-engineering to extract cleaned up event_prices
# This is raw data that leads to a premium dataset

,event_date,entity,event_title,event_price
0,2026-08-31T20:00:00.000000000+00:00,Hawthorne Theatre,"Brenn! - Hawthorne Theatre - Portland, OR",$25 ADV
1,2026-08-31T20:00:00.000000000+00:00,Brenn!,"Brenn! - Hawthorne Theatre - Portland, OR",$25 ADV
2,2026-08-31T20:00:00.000000000+00:00,Tom Siletto,"Brenn! - Hawthorne Theatre - Portland, OR",$25 ADV
3,2026-08-31T09:00:00.000000000+00:00,Nordia House,"Community Groups: Dancing, Nordic Knitting Cir...",FREE (Donations appreciated)
4,2026-08-31T09:00:00.000000000+00:00,Nordic Northwest,"Community Groups: Dancing, Nordic Knitting Cir...",FREE (Donations appreciated)
5,2026-08-31T09:00:00.000000000+00:00,Norske Runddansere,"Community Groups: Dancing, Nordic Knitting Cir...",FREE (Donations appreciated)
6,2026-08-31T08:00:00.000000000+00:00,Providence Portland Medical Center,"Portland, OR - Spinning Babies® Integration Wo...",$250
7,2026-08-31T08:00:00.000000000+00:00,Spinning Babies®,"Portland, OR - Spinning Babies® Integration Wo...",$250
8,2026-08-31T08:00:00.000000000+00:00,Nikki Zerfas,"Portland, OR - Spinning Babies® Integration Wo...",$250
9,2026-08-30T22:00:00.000000000+00:00,Sinferno Cabaret,SINFERNO CABARET - Dante's Live,$13.91 - $313.61


In [50]:
df.shape

(2086, 4)

## Entity Event Prices

Connect entities to specific events and their advertised prices. Use `all` to search every location.

```python
location = "Oregon"
start_date = "2026-08-01"
end_date = "2026-08-31"
```

### Who should use it

* Venues and event promoters
* Artists, managers, and booking agents
* Ticketing and event platforms
* Marketing and event agencies
* Market researchers and OSINT analysts

### Why it matters

Instagram may show a price inside an individual post, but it cannot easily connect pricing patterns to specific venues, artists, or organizations.

Customers can compare how entities price events, benchmark competitors, identify free and premium offerings, and understand how pricing changes across participants and markets.


# tool 7: Entities Collaborations (Location Search)

In [55]:
# genres = [
#     "alt-country",
#     "alternative country"
# ]

# pass in no genres to search for all genres
genres=[]

cypher_query = """ 
 
MATCH (entity:Entity)-[r1]-(event:Event)-[r2]-(date:Date) 
 
WHERE ( 
    toLower($location) = 'all' 
    OR EXISTS { 
        MATCH (event)-[]-(l:Location) 
        WHERE toLower(l.name) = toLower($location) 
    } 
) 
AND date(date.date) >= date($start_date) 
AND date(date.date) <= date($end_date) 
AND (
    size($genres) = 0
    OR any(
        genre IN $genres
        WHERE toLower(event.event_text) CONTAINS toLower(genre)
    )
)
 
RETURN DISTINCT 
    date.date AS event_date, 
    entity.name AS entity, 
    event.event_title AS event_title 
 
ORDER BY event_date DESC, event_title 
 
""" 
 
with driver.session() as session: 
    data = json.loads(json.dumps(
        session.run(
            cypher_query,
            location=location,
            start_date=start_date,
            end_date=end_date,
            genres=genres or [],
        ).data(),
        default=str,
    )) 
 
df = pd.DataFrame(data) 
df.head(10) 
 
# I will use this as raw material to create the temporal entity collaboration graph

,event_date,entity,event_title
0,2026-08-31,Hostel Cafe,*FREE* Live Music & Open Mic
1,2026-08-31,Sonny Hess,2026 Music Mondays
2,2026-08-31,Trent Beaver,2026 Music Mondays
3,2026-08-31,Norman Sylvester,2026 Music Mondays
4,2026-08-31,Eldon T. Jones,2026 Music Mondays
5,2026-08-31,Nola Brass,2026 Music Mondays
6,2026-08-31,Coloso,2026 Music Mondays
7,2026-08-31,Johnny Wheels,2026 Music Mondays
8,2026-08-31,45 Away,2026 Music Mondays
9,2026-08-31,MHCC Jazz Combo,2026 Music Mondays


In [56]:
df.shape

(8300, 3)

## Entity Collaborations

Build a temporal collaboration network from entities connected through shared events. The date window can cover the past, present, or future. Use `all` to build the complete network.

```python
location = "all"
start_date = "2025-01-01"
end_date = "2027-12-31"
```

### Who should use it

* Booking agents and talent managers
* Venues, promoters, and festival organizers
* Music and event marketing companies
* Media and entertainment intelligence firms
* OSINT analysts and network scientists

### Why it matters

Instagram shows isolated posts. It does not reveal the network connecting artists, venues, promoters, organizations, and events over time.

This dataset becomes an edge list for building a collaboration graph. Customers can identify recurring partnerships, influential connectors, communities, emerging relationships, and future collaborations already scheduled to happen.

This is historical, current, and forward-looking network intelligence built from the event market itself.


# Tool 8: Websites

In [63]:
# genres = [
#     "alt-country",
#     "alternative country"
# ]

# pass in no genres to search for all genres
genres=[]

cypher_query = """

MATCH (website:Website)-[]-(event:Event)

WHERE (
    toLower($location) = 'all'
    OR EXISTS {
        MATCH (event)-[]-(l:Location)
        WHERE toLower(l.name) = toLower($location)
    }
)
AND event.event_dates[0] IS NOT NULL
AND date(event.event_dates[0]) >= date($start_date)
AND date(event.event_dates[0]) <= date($end_date)
AND (
    size($genres) = 0
    OR any(
        genre IN $genres
        WHERE toLower(event.event_text) CONTAINS toLower(genre)
    )
)

RETURN
    event.event_dates[0] AS event_date,
    website.domain AS website,
    count(DISTINCT event) AS event_count

ORDER BY event_date DESC
LIMIT 10000

"""

with driver.session() as session:
    data = json.loads(json.dumps(
        session.run(
            cypher_query,
            location=location,
            start_date=start_date,
            end_date=end_date,
            genres=genres or [],
        ).data(),
        default=str,
    ))

df = pd.DataFrame(data)
df.head(10)

,event_date,website,event_count
0,2026-08-31T20:00:00.000000000+00:00,allevents.in,2
1,2026-08-31T20:00:00.000000000+00:00,bandsintown.com,1
2,2026-08-31T20:00:00.000000000+00:00,hawthornetheatre.com,1
3,2026-08-31T19:00:00.000000000+00:00,allevents.in,3
4,2026-08-31T19:00:00.000000000+00:00,mississippipizza.com,1
5,2026-08-31T18:50:00.000000000+00:00,allevents.in,1
6,2026-08-31T18:30:00.000000000+00:00,allevents.in,1
7,2026-08-31T18:00:00.000000000+00:00,kmun.org,1
8,2026-08-31T18:00:00.000000000+00:00,westsideqrc.org,1
9,2026-08-31T17:30:00.000000000+00:00,allevents.in,1


In [64]:
df.shape

(1217, 3)

## Websites

Track websites publishing events within a location and date range. Use `all` to analyze websites across every location.

```python
location = "Oregon"
start_date = "2026-08-01"
end_date = "2026-08-31"
```

### Who should use it

* Media companies and local newsrooms
* Marketing and public relations agencies
* Event platforms and promoters
* Tourism organizations
* Competitive intelligence and OSINT teams

### Why it matters

Instagram shows social posts. It does not reveal the broader network of news sites, event calendars, venue websites, and community publishers distributing event information.

Customers can identify active event publishers, measure publishing volume, discover new distribution channels, compare source activity across markets, and find websites competitors may already be using.


# Tool 9: Website Entities

In [65]:
# genres = [
#     "alt-country",
#     "alternative country"
# ]

# pass in no genres to search for all genres
genres=[]

cypher_query = """

MATCH (website:Website)-[]-(event:Event)-[:MENTIONS]-(entity:Entity)

WHERE (
    toLower($location) = 'all'
    OR EXISTS {
        MATCH (event)-[]-(l:Location)
        WHERE toLower(l.name) = toLower($location)
    }
)
AND event.event_dates[0] IS NOT NULL
AND date(event.event_dates[0]) >= date($start_date)
AND date(event.event_dates[0]) <= date($end_date)
AND (
    size($genres) = 0
    OR any(
        genre IN $genres
        WHERE toLower(event.event_text) CONTAINS toLower(genre)
    )
)

RETURN DISTINCT
    event.event_dates[0] AS event_date,
    website.domain AS website,
    entity.name AS entity

ORDER BY event_date DESC
LIMIT 10000

"""

with driver.session() as session:
    data = json.loads(json.dumps(
        session.run(
            cypher_query,
            location=location,
            start_date=start_date,
            end_date=end_date,
            genres=genres or [],
        ).data(),
        default=str,
    ))

df = pd.DataFrame(data)
df.head(10)

,event_date,website,entity
0,2026-08-31T20:00:00.000000000+00:00,allevents.in,Hawthorne Theatre
1,2026-08-31T20:00:00.000000000+00:00,allevents.in,Brenn!
2,2026-08-31T20:00:00.000000000+00:00,allevents.in,Tom Siletto
3,2026-08-31T20:00:00.000000000+00:00,bandsintown.com,Hawthorne Theatre
4,2026-08-31T20:00:00.000000000+00:00,bandsintown.com,Brenn
5,2026-08-31T20:00:00.000000000+00:00,hawthornetheatre.com,Hawthorne Theatre
6,2026-08-31T20:00:00.000000000+00:00,hawthornetheatre.com,BRENN!
7,2026-08-31T20:00:00.000000000+00:00,allevents.in,McMenamins Crystal Ballroom
8,2026-08-31T20:00:00.000000000+00:00,allevents.in,Thee Oh Sees
9,2026-08-31T19:00:00.000000000+00:00,allevents.in,Atlantis Lounge


In [66]:
df.shape

(3904, 3)

## Website Entities

Connect websites to the entities appearing in the events they publish. Use `all` to analyze relationships across every location.

```python
location = "Oregon"
start_date = "2026-08-01"
end_date = "2026-08-31"
```

### Who should use it

* Marketing and public relations agencies
* Artists, managers, and booking agents
* Venues and event promoters
* Media intelligence companies
* Competitive intelligence and OSINT teams

### Why it matters

Instagram shows who posts on Instagram. It does not reveal which websites publish information about particular artists, venues, and organizations.

Customers can identify who covers or amplifies each entity, discover media relationships, find outreach targets, compare competitor visibility, and locate gaps where an entity is receiving little coverage.


# Tool 10: Website Events

In [67]:
# genres = [
#     "alt-country",
#     "alternative country"
# ]

# pass in no genres to search for all genres
genres=[]

cypher_query = """

MATCH (website:Website)-[]-(event:Event)

WHERE (
    toLower($location) = 'all'
    OR EXISTS {
        MATCH (event)-[]-(l:Location)
        WHERE toLower(l.name) = toLower($location)
    }
)
AND event.event_dates[0] IS NOT NULL
AND date(event.event_dates[0]) >= date($start_date)
AND date(event.event_dates[0]) <= date($end_date)
AND (
    size($genres) = 0
    OR any(
        genre IN $genres
        WHERE toLower(event.event_text) CONTAINS toLower(genre)
    )
)

RETURN DISTINCT
    event.event_dates[0] AS event_date,
    website.domain AS website,
    event.event_title AS event_title

ORDER BY event_date DESC
LIMIT 10000

"""

with driver.session() as session:
    data = json.loads(json.dumps(
        session.run(
            cypher_query,
            location=location,
            start_date=start_date,
            end_date=end_date,
            genres=genres or [],
        ).data(),
        default=str,
    ))

df = pd.DataFrame(data)
df.head(10)

,event_date,website,event_title
0,2026-08-31T20:00:00.000000000+00:00,allevents.in,"Brenn! - Hawthorne Theatre - Portland, OR"
1,2026-08-31T20:00:00.000000000+00:00,allevents.in,Thee Oh Sees in Portland
2,2026-08-31T20:00:00.000000000+00:00,bandsintown.com,Brenn!
3,2026-08-31T20:00:00.000000000+00:00,hawthornetheatre.com,BRENN! - AMATEUR AT BEST WORLD TOUR
4,2026-08-31T19:00:00.000000000+00:00,allevents.in,Covers Only (Open Mic)
5,2026-08-31T19:00:00.000000000+00:00,allevents.in,Reading: Melissa Bird
6,2026-08-31T19:00:00.000000000+00:00,allevents.in,MODERN KUNDALINI (Monday Evenings)
7,2026-08-31T19:00:00.000000000+00:00,mississippipizza.com,FREE! Covers Only
8,2026-08-31T18:50:00.000000000+00:00,allevents.in,Solsara Practice Night
9,2026-08-31T18:30:00.000000000+00:00,allevents.in,*FREE* Live Music & Open Mic


In [68]:
df.shape

(1495, 3)

## Website Events

Connect websites to the specific events they publish. Use `all` to analyze event coverage across every location.

```python
location = "Oregon"
start_date = "2026-08-01"
end_date = "2026-08-31"
```

### Who should use it

* Media companies and local newsrooms
* Marketing and public relations agencies
* Venues and event promoters
* Event platforms and tourism organizations
* Competitive intelligence and OSINT teams

### Why it matters

Instagram provides scattered promotion without a clear view of which websites are covering which events.

Customers can track event coverage, identify active publishers, discover distribution channels, compare coverage between events, and find opportunities to place events on websites where competitors are already visible.


# Tool 11: Country Counts

In [83]:
countries = [country.name for country in list(pycountry.countries)]

# genres = [
#     "alt-country",
#     "alternative country"
# ]

# pass in no genres to search for all genres
genres=[]

cypher_query = """

UNWIND $countries AS country

MATCH (location:Location {name: country})-[]-(event:Event)

WHERE event.event_dates[0] IS NOT NULL
AND date(event.event_dates[0]) >= date($start_date)
AND date(event.event_dates[0]) <= date($end_date)
AND (
    size($genres) = 0
    OR any(
        genre IN $genres
        WHERE toLower(event.event_text) CONTAINS toLower(genre)
    )
)

RETURN
    country,
    count(DISTINCT event) AS event_count

ORDER BY event_count DESC

"""

with driver.session() as session:
    data = json.loads(json.dumps(
        session.run(
            cypher_query,
            countries=countries,
            start_date=start_date,
            end_date=end_date,
            genres=genres or [],
        ).data(),
        default=str,
    ))

df = pd.DataFrame(data)
df

,country,event_count
0,United States,6022
1,Germany,1306
2,United Kingdom,787
3,Canada,578
4,Hungary,195
...,...,...
81,Pakistan,1
82,French Polynesia,1
83,San Marino,1
84,Slovakia,1


## Country Event Coverage

Measure event activity across countries and compare where events are being discovered and published. Filter by genre to analyze specific music, cultural, or event ecosystems.

```python
start_date = "2026-08-01"
end_date = "2026-08-31"

genres = []
```

### Who should use it

* Event platforms and tourism organizations
* Media and entertainment companies
* Artists, promoters, and booking agencies
* Market research and competitive intelligence teams
* OSINT investigators and data analysts

### Why it matters

Event activity is fragmented across thousands of local sources, making international markets difficult to compare.

Customers can measure event activity by country, identify active and underserved markets, compare geographic demand and coverage, analyze specific genres across countries, and discover opportunities for expansion, touring, promotion, and market research.

In [84]:
#%pip install pycountry